<a href="https://colab.research.google.com/github/avsnrakhila-2007/ML_lab_Programs/blob/main/ml_skill_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

from google.colab import files

print("Upload your 50K placement prediction CSV file")

uploaded = files.upload()

filename = list(uploaded.keys())[0]

print("\nUploaded file:", filename)

df = pd.read_csv(filename)

print("\n============================================================")
print("DATASET LOADED")
print("============================================================")

print("Dataset shape:", df.shape)

print("\nColumn names:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

before = len(df)

df = df.drop_duplicates()

after = len(df)

print("\n============================================================")
print("STEP 1: REMOVE DUPLICATES")
print("============================================================")

print("Rows before:", before)
print("Rows after :", after)
print("Duplicates removed:", before - after)

print("\n============================================================")
print("STEP 2: MISSING VALUES")
print("============================================================")

print("\nMissing values BEFORE:")

print(df.isna().sum())

print(
    "\nTotal missing values BEFORE:",
    df.isna().sum().sum()
)

NUM = [
    "CGPA",
    "AttendancePercent",
    "HistoryOfBacklogs",
    "Internships",
    "Projects",
    "Workshops",
    "Certifications",
    "Publications",
    "AptitudeTestScore",
    "SoftSkillsRating",
    "CodingTestScore",
    "MockInterviewScore",
    "ExtraCurricular"
]

CAT = [
    "Gender",
    "City",
    "CollegeTier",
    "Stream",
    "Specialisation",
    "Hostel",
    "CGPA_Tier"
]

if df["HistoryOfBacklogs"].dtype == "object":

    df["HistoryOfBacklogs"] = df[
        "HistoryOfBacklogs"
    ].map({
        "Yes": 1,
        "No": 0
    })

for col in NUM:

    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    df[col] = df[col].fillna(
        df[col].median()
    )

for col in CAT:

    df[col] = df[col].fillna(
        df[col].mode()[0]
    )

print("\nMissing values AFTER filling:")

print(df[NUM + CAT].isna().sum())

print(
    "\nTotal missing values AFTER:",
    df[NUM + CAT].isna().sum().sum()
)

print("\n============================================================")
print("STEP 3: CGPA OUTLIER DETECTION")
print("============================================================")

q1 = df["CGPA"].quantile(0.25)

q3 = df["CGPA"].quantile(0.75)

iqr = q3 - q1

lo = q1 - 1.5 * iqr

hi = q3 + 1.5 * iqr

outlier_mask = (
    (df["CGPA"] < lo) |
    (df["CGPA"] > hi)
)

outlier_count = int(
    outlier_mask.sum()
)

print("Q1:", round(q1, 2))

print("Q3:", round(q3, 2))

print("IQR:", round(iqr, 2))

print("Lower fence:", round(lo, 2))

print("Upper fence:", round(hi, 2))

print(
    "CGPA outliers:",
    outlier_count
)

df["CGPA"] = df["CGPA"].clip(
    lower=lo,
    upper=hi
)

print("\nCGPA outliers clipped successfully.")

prep = ColumnTransformer(

    transformers=[

        (
            "num",

            Pipeline([
                (
                    "imp",
                    SimpleImputer(
                        strategy="median"
                    )
                ),

                (
                    "sc",
                    StandardScaler()
                )
            ]),

            NUM
        ),

        (
            "cat",

            Pipeline([
                (
                    "imp",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),

                (
                    "oh",
                    OneHotEncoder(
                        handle_unknown="ignore"
                    )
                )
            ]),

            CAT
        )
    ]
)

X = df[NUM + CAT]

y = df["PlacementStatus"]

X_processed = prep.fit_transform(X)

print("\n============================================================")
print("STEP 4: SCALING + ONE-HOT ENCODING")
print("============================================================")

print(
    "Original feature count:",
    len(NUM) + len(CAT)
)

print(
    "Processed shape:",
    X_processed.shape
)

feature_names = (
    prep.get_feature_names_out()
)

print("\nFirst 20 processed feature names:")

print(
    list(feature_names[:20])
)

numeric_part = X_processed[:, :len(NUM)]

if hasattr(
    numeric_part,
    "toarray"
):

    numeric_part = numeric_part.toarray()

means = numeric_part.mean(
    axis=0
)

print(
    "\nNumeric means after scaling "
    "(should be approximately 0):"
)

print(
    np.round(
        means,
        3
    )
)

print("\n============================================================")
print("PLACEMENT STATUS DISTRIBUTION")
print("============================================================")

print(
    df["PlacementStatus"].value_counts()
)

df.to_csv(
    "placement_clean.csv",
    index=False
)

print("\n============================================================")
print("CLEAN DATASET SAVED")
print("============================================================")

print(
    "File created: placement_clean.csv"
)

print(
    "Final dataset shape:",
    df.shape
)

import joblib

joblib.dump(
    prep,
    "placement_preprocessor.joblib"
)

print(
    "Preprocessor saved as: "
    "placement_preprocessor.joblib"
)

files.download(
    "placement_clean.csv"
)

print("\n============================================================")
print("EXPERIMENT 2 COMPLETED SUCCESSFULLY!")
print("============================================================")